### Overall metric comparison

In [ ]:
import pandas as pd
import glob
import numpy as np
import matplotlib.pyplot as plt
import plotnine as gg
import scipy.stats as stats

files = glob.glob(
    "/workspace/experiments/06052026_cellbox_noise/results/*/metrics/perturbation_metrics.csv"
)
print(files)
all_dfs = []
for file in files:
    tag_name = file.split("/")[-3]
    df = pd.read_csv(file).assign(tag=tag_name)
    all_dfs.append(df)
all_df = pd.concat(all_dfs, ignore_index=True)

In [ ]:
metrics_of_interest = [
    "lfc_pearson_r",
    "lfc_pearson_r_top20_degs",
    "lfc_pearson_r_top100_degs",
    "auroc_top20_degs",
    "auroc_top100_degs",
]

In [ ]:
all_df.groupby("tag")[metrics_of_interest].mean()

In [ ]:
###

### Scatter plots for individual perturbations

In [ ]:
import scanpy as sc

adata_pred = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/results/linear_residual/adata_pred.h5ad"
)

adata_pred_nb = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/results/nb_residual/adata_pred.h5ad"
)


adata_pred_nb_ds = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/results/nb_ds_residual/adata_pred.h5ad"
)


adata_test = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/data/adata_tf_test_0.h5ad"
)
adata_control = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/data/adata_control.h5ad"
)
adata_train = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/data/adata_train_0.h5ad"
)
adata_tf_train = sc.read_h5ad(
    "/workspace/experiments/06052026_cellbox_noise/data/adata_tf_train_0.h5ad"
)

In [ ]:
def compute_lfcs(adata_pred):
    res = []
    for target in adata_test.obs["target"].unique():
        lfc_pred = adata_pred[adata_pred.obs["target"] == target].X.mean(
            axis=0
        ) - adata_control.layers["log1p"].mean(axis=0)
        lfc_true = adata_test[adata_test.obs["target"] == target].layers["log1p"].mean(
            axis=0
        ) - adata_control.layers["log1p"].mean(axis=0)

        df = pd.DataFrame(
            {
                "lfc_pred": lfc_pred,
                "lfc_true": lfc_true,
                "gene": adata_test.var_names,
            }
        ).assign(target=target)
        res.append(df)
    res = pd.concat(res)
    return res

In [ ]:
res_linear = compute_lfcs(adata_pred).assign(model="gaussian")
res_nb = compute_lfcs(adata_pred_nb).assign(model="nb")
res_nb_ds = compute_lfcs(adata_pred_nb_ds).assign(model="nb_ds")

In [ ]:
(
    all_df.loc[lambda x: x["tag"] == "linear_residual"]
    .groupby(["tag", "perturbation"])["lfc_pearson_r_top20_degs"]
    .mean()
    .sort_values(ascending=False)
)

In [ ]:
heldout_perts = res_nb["target"].unique()

In [ ]:
plot_df = res_linear.query("target == 'laci'").copy()

(
    gg.ggplot(plot_df, gg.aes(x="lfc_true", y="lfc_pred"))
    + gg.geom_point()
    + gg.theme_bw()
    + gg.labs(
        x="True LFC",
        y="Predicted LFC",
        title="LFC predictions vs true values for lacI KD",
    )
)

In [ ]:
res_nb.query("target == 'laci'").sort_values("lfc_pred")

In [ ]:
heldout_perts = res_linear["target"].unique()
for pert in heldout_perts:
    plot_df = res_linear.query(f"target == '{pert}'").copy()
    p = (
        gg.ggplot(plot_df, gg.aes(x="lfc_true", y="lfc_pred"))
        + gg.geom_point()
        + gg.theme_bw()
        + gg.labs(
            x="True LFC",
            y="Predicted LFC",
            title=f"LFC predictions vs true values for {pert} KD",
        )
        + gg.xlim(-2.5, 2.5)
        # + gg.ylim(-2.5, 2.5)
    )
    display(p)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

kmeans = GaussianMixture(n_components=6, covariance_type="full").fit(
    plot_df[["lfc_true", "lfc_pred"]], plot_df["gene"]
)
plot_df["cluster"] = kmeans.predict(plot_df[["lfc_true", "lfc_pred"]])

(
    gg.ggplot(plot_df, gg.aes(x="lfc_true", y="lfc_pred", color="factor(cluster)"))
    + gg.geom_point()
    + gg.theme_bw()
)

In [ ]:
well_predicted_genes = plot_df.loc[lambda x: x["cluster"] == 4]["gene"].values
poorly_predicted_genes = plot_df.loc[lambda x: x["cluster"] == 1]["gene"].values

In [ ]:
amat = np.load("/workspace/experiments/06052026_cellbox_noise/data/amask.npy")
amat_df = pd.DataFrame(
    amat, columns=adata_pred_nb.var_names, index=adata_pred_nb.var_names
)
n_regs_per_gene = amat_df.sum(axis=1)
n_regs_per_gene

In [ ]:
n_regs_well_predicted = n_regs_per_gene.reindex(well_predicted_genes).fillna(0)
n_regs_poorly_predicted = n_regs_per_gene.reindex(poorly_predicted_genes).fillna(0)

bins = np.arange(0, 20, 1)
plt.hist(n_regs_well_predicted, bins=bins, alpha=0.5, label="well predicted")
plt.hist(n_regs_poorly_predicted, bins=bins, alpha=0.5, label="poorly predicted")
plt.legend()
plt.show()

In [ ]:
n_regs_well_predicted.describe()

In [ ]:
n_regs_poorly_predicted.describe()

In [ ]:
res_nb["lfc_true"].hist(bins=50)

In [ ]:
top_downregulated = (
    res_nb.loc[lambda x: x["target"] == "laci"]
    .sort_values("lfc_true")
    .head(25)[["lfc_true", "gene"]]
)
top_downregulated

In [ ]:
downregulated_genes = top_downregulated["gene"].values

In [ ]:
res_nb.loc[lambda x: x["gene"].isin(downregulated_genes)]["lfc_true"].hist()

In [ ]:
import jax.nn as jnn


def describe_gene(gene, model, perturb_gene=None):
    """
    Print the learned parameters for `gene` and trace through the steady-state
    dynamics, optionally under a KD of `perturb_gene`.

    Columns for regulators:
      A[i,j]      — learned weight (positive = activating, negative = repressing
                    in expression space; note: sign is learned from co-expression,
                    not from ground-truth biology)
      z_ctrl      — normalised expression of regulator at control mean (= 0 by
                    construction, since x_mean_ is fit to control)
      z_kd        — normalised expression of regulator after KD (only shown when
                    perturb_gene is that regulator)
      Δpreact     — A[i,j] * (z_kd - z_ctrl): change in preactivation of `gene`
                    contributed by this regulator under the perturbation
    """
    params = model.state.params
    gene_names = list(model.adata.var_names)
    idx = gene_names.index(gene)

    eps = float(np.exp(params["epsilon_"][idx]))
    b = float(params["b_"][idx])
    xmean = float(params["x_mean_"][idx])
    xstd = float(params["x_std_"][idx])

    ss_ctrl = eps * float(jnn.sigmoid(np.array(b)))
    lfc_bias = ss_ctrl - xmean

    print(f"{'='*52}")
    print(f"  Gene : {gene}")
    print(f"{'='*52}")
    print(f"  epsilon (sigmoid ceiling, log-CP10K) : {eps:.4f}")
    print(f"  b       (bias)                       : {b:.4f}")
    print(f"  x_mean  (ctrl log-CP10K mean)         : {xmean:.4f}")
    print(f"  x_std   (ctrl log-CP10K std)          : {xstd:.4f}")
    print(f"  ss @ ctrl  = eps * σ(b)              : {ss_ctrl:.4f}")
    print(f"  constant LFC bias (ss - x_mean)       : {lfc_bias:+.4f}")
    print()

    amat = model.get_Amat()
    regs = amat.loc[gene]
    regs = regs[regs != 0].sort_values(key=abs, ascending=False)

    if len(regs) == 0:
        print("  No regulators — output is the constant ss @ ctrl above.")
        return

    # Preactivation at control: normalize(mu_ctrl) = 0 by construction,
    # so preact_ctrl = b (all regulator contributions are zero).
    preact_ctrl = b

    # Optional: compute regulator's expression under KD
    kd_z = {}  # regulator name → z-score under KD
    if perturb_gene is not None:
        kidx = gene_names.index(perturb_gene)
        eps_k = float(np.exp(params["epsilon_"][kidx]))
        b_k = float(params["b_"][kidx])
        xm_k = float(params["x_mean_"][kidx])
        xs_k = float(params["x_std_"][kidx])
        # First-order approximation: x_kd ≈ eps_k * σ(b_k - 10)
        x_kd = eps_k * float(jnn.sigmoid(np.array(b_k - 10.0)))
        z_kd = (x_kd - xm_k) / (xs_k + 1e-8)
        kd_z[perturb_gene] = z_kd

    print(f"  Regulators of {gene} ({len(regs)}):")
    hdr = f"  {'regulator':>12}  {'A[i,j]':>8}  {'z_ctrl':>7}"
    if perturb_gene:
        hdr += f"  {'z_kd':>7}  {'Δpreact':>8}"
    print(hdr)
    print("  " + "-" * (len(hdr) - 2))

    delta_preact_total = 0.0
    for reg, w in regs.items():
        z_c = 0.0  # normalized ctrl mean is zero by definition
        row = f"  {reg:>12}  {w:8.4f}  {z_c:7.4f}"
        if perturb_gene:
            z_k = kd_z.get(reg, z_c)  # other regulators unchanged at ctrl
            d = float(w) * (z_k - z_c)
            delta_preact_total += d
            row += f"  {z_k:7.4f}  {d:+8.4f}"
        print(row)

    print()
    print(f"  Preactivation @ ctrl  = b = {preact_ctrl:.4f}")
    print(f"  Predict @ ctrl        = {eps:.4f} * σ({preact_ctrl:.4f}) = {ss_ctrl:.4f}")
    if perturb_gene:
        preact_kd = preact_ctrl + delta_preact_total
        ss_kd = eps * float(jnn.sigmoid(np.array(preact_kd)))
        print()
        print(f"  Under {perturb_gene} KD (first-order, other regs at ctrl):")
        print(f"    Δpreact from {perturb_gene} : {delta_preact_total:+.4f}")
        print(f"    Preactivation @ KD : {preact_kd:.4f}")
        print(f"    Predict @ KD       : {eps:.4f} * σ({preact_kd:.4f}) = {ss_kd:.4f}")
        print(
            f"    Predicted ΔLFC     : {ss_kd - ss_ctrl:+.4f}  (= pred_kd - pred_ctrl)"
        )

In [ ]:
describe_gene("laca", model, perturb_gene="laci")
print()
describe_gene("laci", model)